# Assignment 2: UNO Game AI
**Name:** Afraz Ahmad  
**Roll No:** i242598  
**GitHub:** https://github.com/AfrazAhmad11/UNO_GameAI

In [29]:
import random
import copy
import tkinter as tk
from tkinter import messagebox, simpledialog

In [30]:
class Card:
    """
    Represents a single UNO card.
    color: Red, Blue, Green, Yellow
    value: 0-9 or 'Skip'
    """
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

    def __eq__(self, other):
        # Two cards are equal if same color and value
        return self.color == other.color and self.value == other.value

In [31]:
def generate_deck():
    """
    Generates a shuffled UNO deck.
    Contains: Red/Blue/Green/Yellow 0-9 + 2 Skip cards per color
    """
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []

    # Add number cards 0-9 for each color
    for color in colors:
        for number in range(10):
            deck.append(Card(color, number))

    # Add 2 Skip cards per color
    for color in colors:
        deck.append(Card(color, 'Skip'))
        deck.append(Card(color, 'Skip'))

    # Shuffle the deck
    random.shuffle(deck)
    return deck

# Test deck
deck = generate_deck()
print(f"Total cards in deck: {len(deck)}")
print(f"First 5 cards: {deck[:5]}")

Total cards in deck: 48
First 5 cards: [Yellow 4, Red 0, Yellow 5, Yellow 2, Blue 4]


In [32]:
# Game State + Deal Card

In [33]:
def initialize_game():
    """
    Sets up the initial game state.
    Each player gets 5 cards. Top card is revealed.
    """
    deck = generate_deck()

    # Deal 5 cards to each player
    p1_hand = [deck.pop() for _ in range(5)]  # Player 1 - Minimax (Defensive)
    p2_hand = [deck.pop() for _ in range(5)]  # Player 2 - Expectimax (Offensive)
    p3_hand = [deck.pop() for _ in range(5)]  # Player 3 - User/AI

    # Pick a top card (must be a number card, not Skip)
    top_card = None
    while top_card is None:
        card = deck.pop()
        if card.value != 'Skip':
            top_card = card
        else:
            deck.insert(0, card)  # Put Skip back at bottom

    # Game state dictionary
    state = {
        'p1_hand': p1_hand,   # Player 1 hand
        'p2_hand': p2_hand,   # Player 2 hand
        'p3_hand': p3_hand,   # Player 3 hand
        'top_card': top_card, # Current top card on discard pile
        'deck': deck,         # Remaining draw deck
        'skip_next': None     # Tracks who is skipped
    }
    return state

# Test initialization
state = initialize_game()
print(f"Top Card: {state['top_card']}")
print(f"P1 Hand: {state['p1_hand']}")
print(f"P2 Hand: {state['p2_hand']}")
print(f"P3 Hand: {state['p3_hand']}")
print(f"Remaining deck: {len(state['deck'])} cards")

Top Card: Blue 8
P1 Hand: [Blue Skip, Yellow 5, Red 1, Blue 3, Green 9]
P2 Hand: [Blue 7, Blue 4, Blue Skip, Blue 0, Yellow 7]
P3 Hand: [Red 3, Green 4, Green 7, Green Skip, Red 7]
Remaining deck: 32 cards


In [34]:
# Legal Move Generator

In [35]:
def get_valid_moves(hand, top_card):
    """
    Returns list of valid cards a player can play.
    Rule: Card must match top card's color OR number/value.
    """
    valid = []
    for card in hand:
        # Same color OR same number/value
        if card.color == top_card.color or card.value == top_card.value:
            valid.append(card)
    return valid

# Test
top = Card('Red', 5)
hand = [Card('Red', 3), Card('Blue', 5), Card('Green', 7), Card('Red', 'Skip'), Card('Yellow', 2)]
print(f"Top Card: {top}")
print(f"Valid moves: {get_valid_moves(hand, top)}")

Top Card: Red 5
Valid moves: [Red 3, Blue 5, Red Skip]


In [36]:
# State Transition /(apply move)

In [37]:
def apply_move(state, player, card):
    """
    Applies a move to the state and returns a new state.
    player: 'p1', 'p2', or 'p3'
    card: Card object or None (means draw)
    """
    # Deep copy so original state is not modified
    new_state = copy.deepcopy(state)
    hand_key = f'{player}_hand'

    if card is None:
        # DRAW: player draws 1 card from deck
        if new_state['deck']:
            drawn = new_state['deck'].pop()
            new_state[hand_key].append(drawn)
    else:
        # PLAY: remove card from hand, set as top card
        new_state[hand_key] = [c for c in new_state[hand_key]
                                if not (c.color == card.color and c.value == card.value)]
        # Remove only one copy
        original = state[hand_key]
        new_hand = copy.deepcopy(original)
        for i, c in enumerate(new_hand):
            if c.color == card.color and c.value == card.value:
                new_hand.pop(i)
                break
        new_state[hand_key] = new_hand
        new_state['top_card'] = copy.deepcopy(card)

        # Handle Skip card
        if card.value == 'Skip':
            new_state['skip_next'] = True
        else:
            new_state['skip_next'] = False

    return new_state

print("apply_move function defined successfully.")

apply_move function defined successfully.


---
## Cell 7: Evaluation Function

### Formula:
**Score = 50 − 5(C_AI) + 2(C_opp) + 3(S)**

- **C_AI**: Number of cards in current player's hand (fewer = better)
- **C_opp**: Average cards held by opponents (more opponent cards = better for us)
- **S**: Number of Skip cards in hand (more skips = more control)

### Weight Tuning:
- **Defensive (P1)**: Higher penalty for own cards, higher reward for Skip (control)
- **Offensive (P2)**: Higher reward for opponent cards, focus on shedding own cards fast

In [38]:
def evaluate(state, player, strategy='defensive'):
    """
    Evaluation function for the given player.
    strategy: 'defensive' (P1 - Minimax) or 'offensive' (P2 - Expectimax)

    Base formula: Score = 50 - 5*C_AI + 2*C_opp + 3*S
    Weights are tuned per strategy.
    """
    hand_key = f'{player}_hand'
    players = ['p1', 'p2', 'p3']
    opponents = [p for p in players if p != player]

    # Cards in AI hand
    c_ai = len(state[hand_key])

    # Average cards in opponents' hands
    c_opp = sum(len(state[f'{p}_hand']) for p in opponents) / len(opponents)

    # Skip cards in hand
    s = sum(1 for card in state[hand_key] if card.value == 'Skip')

    if strategy == 'defensive':
        # Defensive: penalize own cards more, reward skips more
        # Focuses on not losing rather than winning fast
        w_ai   = 6   # Higher penalty for own cards
        w_opp  = 2   # Normal weight for opponents
        w_skip = 4   # Higher reward for skip (control)
        score = 50 - w_ai * c_ai + w_opp * c_opp + w_skip * s

    elif strategy == 'offensive':
        # Offensive: reward shedding cards fast, also reward hurting opponents
        # Focuses on winning quickly
        w_ai   = 5   # Normal penalty for own cards
        w_opp  = 3   # Higher reward for opponent cards (want them stuck)
        w_skip = 2   # Lower skip reward (less focused on control)
        score = 50 - w_ai * c_ai + w_opp * c_opp + w_skip * s

    else:
        # Default baseline formula
        score = 50 - 5 * c_ai + 2 * c_opp + 3 * s

    return round(score, 2)

# Test evaluation
state = initialize_game()
print(f"P1 Defensive Score: {evaluate(state, 'p1', 'defensive')}")
print(f"P2 Offensive Score: {evaluate(state, 'p2', 'offensive')}")

P1 Defensive Score: 30.0
P2 Offensive Score: 42.0


In [39]:
# Minimax Algorithm (Player 1 – Defensive)

In [40]:
# Global tree log for game tree printing
minimax_tree_log = []

def minimax(state, depth, is_maximizing, current_player, ai_player, log=None, indent=0):
    """
    Minimax algorithm for Defensive Player (P1).
    - MAX node: AI's turn (maximize score)
    - MIN node: Opponent's turn (minimize AI's score)
    Depth: 3
    """
    hand_key = f'{ai_player}_hand'

    # Terminal: AI has no cards (win) or depth 0
    if len(state[hand_key]) == 0 or depth == 0:
        score = evaluate(state, ai_player, 'defensive')
        if log is not None:
            log.append(' ' * indent + f"[LEAF] depth={depth} score={score}")
        return score, None

    players_order = ['p1', 'p2', 'p3']
    idx = players_order.index(current_player)
    next_player = players_order[(idx + 1) % 3]

    hand_key_curr = f'{current_player}_hand'
    valid_moves = get_valid_moves(state[hand_key_curr], state['top_card'])
    # Add DRAW option
    moves = valid_moves + [None]  # None = draw

    if is_maximizing:
        # MAX node – AI tries to maximize score
        best_score = float('-inf')
        best_move = None
        if log is not None:
            log.append(' ' * indent + f"[MAX] {current_player} | top={state['top_card']} | hand={state[hand_key_curr]}")

        for move in moves:
            new_state = apply_move(state, current_player, move)
            label = str(move) if move else 'DRAW'
            if log is not None:
                log.append(' ' * (indent+2) + f"-> Try: {label}")
            score, _ = minimax(new_state, depth - 1, False, next_player, ai_player, log, indent+4)
            if score > best_score:
                best_score = score
                best_move = move

        return best_score, best_move

    else:
        # MIN node – Opponent tries to minimize AI score
        best_score = float('inf')
        best_move = None
        if log is not None:
            log.append(' ' * indent + f"[MIN] {current_player} | top={state['top_card']} | hand={state[hand_key_curr]}")

        for move in moves:
            new_state = apply_move(state, current_player, move)
            label = str(move) if move else 'DRAW'
            if log is not None:
                log.append(' ' * (indent+2) + f"-> Try: {label}")
            # Next maximizing: only when it's AI's turn again
            is_max_next = (next_player == ai_player)
            score, _ = minimax(new_state, depth - 1, is_max_next, next_player, ai_player, log, indent+4)
            if score < best_score:
                best_score = score
                best_move = move

        return best_score, best_move


def p1_minimax_move(state, log=None):
    """
    Player 1 uses Minimax (Defensive) to pick best move.
    Returns the best card to play, or None to draw.
    """
    score, move = minimax(state, depth=3, is_maximizing=True,
                          current_player='p1', ai_player='p1', log=log)
    return move

print("Minimax algorithm defined.")

Minimax algorithm defined.


In [41]:
# Expectimax Algorithm (Player 2 – Offensive)

In [42]:
expectimax_tree_log = []

def expectimax(state, depth, node_type, current_player, ai_player, log=None, indent=0):
    """
    Expectimax algorithm for Offensive Player (P2).
    node_type: 'max' | 'chance' | 'opponent'
    - MAX node: AI's turn
    - CHANCE node: Draw card (probability-weighted)
    - OPPONENT node: Random legal move by opponent
    Depth: 3
    """
    hand_key = f'{ai_player}_hand'

    # Terminal conditions
    if len(state[hand_key]) == 0 or depth == 0:
        score = evaluate(state, ai_player, 'offensive')
        if log is not None:
            log.append(' ' * indent + f"[LEAF] depth={depth} score={score}")
        return score, None

    players_order = ['p1', 'p2', 'p3']
    idx = players_order.index(current_player)
    next_player = players_order[(idx + 1) % 3]

    hand_key_curr = f'{current_player}_hand'
    valid_moves = get_valid_moves(state[hand_key_curr], state['top_card'])

    if node_type == 'max':
        # MAX node: AI picks best move
        best_score = float('-inf')
        best_move = None
        if log is not None:
            log.append(' ' * indent + f"[MAX] {current_player} | top={state['top_card']} | hand={state[hand_key_curr]}")

        # Try each valid card
        for move in valid_moves:
            new_state = apply_move(state, current_player, move)
            if log is not None:
                log.append(' ' * (indent+2) + f"-> Play: {move}")
            score, _ = expectimax(new_state, depth - 1, 'opponent', next_player, ai_player, log, indent+4)
            if score > best_score:
                best_score = score
                best_move = move

        # Also consider DRAW as chance node
        if log is not None:
            log.append(' ' * (indent+2) + "-> DRAW (Chance Node)")
        draw_score, _ = expectimax(state, depth - 1, 'chance', current_player, ai_player, log, indent+4)
        if draw_score > best_score:
            best_score = draw_score
            best_move = None  # None = draw

        return best_score, best_move

    elif node_type == 'chance':
        # CHANCE node: expected value over all possible drawn cards
        deck = state['deck']
        if not deck:
            score = evaluate(state, ai_player, 'offensive')
            return score, None

        total_cards = len(deck)
        expected_score = 0.0

        if log is not None:
            log.append(' ' * indent + f"[CHANCE] deck size={total_cards}")

        # Group cards by type to compute probability
        unique_cards = {}
        for c in deck:
            key = f"{c.color}_{c.value}"
            unique_cards[key] = unique_cards.get(key, 0) + 1

        for key, count in unique_cards.items():
            prob = count / total_cards  # Probability of drawing this card type
            # Simulate drawing this card
            color, value = key.split('_')
            # value might be a number string
            try:
                value = int(value)
            except ValueError:
                pass  # Keep as 'Skip'

            drawn_card = Card(color, value)
            new_state = copy.deepcopy(state)
            new_state[hand_key_curr].append(drawn_card)

            if log is not None:
                log.append(' ' * (indent+2) + f"P({drawn_card})={prob:.2f}")

            score, _ = expectimax(new_state, depth - 1, 'opponent', next_player, ai_player, log, indent+4)
            expected_score += prob * score

        return round(expected_score, 2), None

    elif node_type == 'opponent':
        # OPPONENT node: opponent picks a random legal move
        if log is not None:
            log.append(' ' * indent + f"[OPP] {current_player} | top={state['top_card']} | hand={state[hand_key_curr]}")

        if not valid_moves:
            # Must draw
            new_state = apply_move(state, current_player, None)
            is_next_max = (next_player == ai_player)
            next_type = 'max' if next_player == ai_player else 'opponent'
            return expectimax(new_state, depth - 1, next_type, next_player, ai_player, log, indent+4)

        # Random move (simulate opponent randomness)
        move = random.choice(valid_moves)
        new_state = apply_move(state, current_player, move)
        if log is not None:
            log.append(' ' * (indent+2) + f"-> Random play: {move}")
        next_type = 'max' if next_player == ai_player else 'opponent'
        return expectimax(new_state, depth - 1, next_type, next_player, ai_player, log, indent+4)


def p2_expectimax_move(state, log=None):
    """
    Player 2 uses Expectimax (Offensive) to pick best move.
    Returns the best card to play, or None to draw.
    Also prints all possible moves with expected scores.
    """
    hand = state['p2_hand']
    top = state['top_card']
    valid = get_valid_moves(hand, top)

    print(f"\nTop card: {top}")
    print(f"P2 hand: {hand}")
    print("P2 decision (All possible decisions at depth 1):")

    best_score = float('-inf')
    best_move = None

    for move in valid:
        new_state = apply_move(state, 'p2', move)
        score = evaluate(new_state, 'p2', 'offensive')
        print(f"  Play: {move} | Expected score: {score}")
        if score > best_score:
            best_score = score
            best_move = move

    # Draw option
    draw_state = apply_move(state, 'p2', None)
    draw_score = evaluate(draw_state, 'p2', 'offensive')
    print(f"  DRAW | Expected score: {draw_score}")
    if draw_score > best_score:
        best_score = draw_score
        best_move = None

    # Full expectimax for actual best move
    _, move = expectimax(state, depth=3, node_type='max',
                         current_player='p2', ai_player='p2', log=log)
    return move

print("Expectimax algorithm defined.")

Expectimax algorithm defined.


In [43]:
#Player 3 Logic (Manual + Simulation

In [44]:
def p3_manual_move(state):
    """
    Manual mode: User picks a card to play.
    Shows valid moves and asks for input.
    """
    hand = state['p3_hand']
    top = state['top_card']
    valid = get_valid_moves(hand, top)

    print(f"\n--- YOUR TURN (Player 3) ---")
    print(f"Top card: {top}")
    print(f"Your hand: {hand}")

    if not valid:
        print("No valid moves. You must draw a card.")
        return None

    print("Valid moves:")
    for i, card in enumerate(valid):
        print(f"  {i}: {card}")
    print(f"  {len(valid)}: DRAW")

    while True:
        try:
            choice = int(input("Enter your choice: "))
            if choice == len(valid):
                return None  # Draw
            elif 0 <= choice < len(valid):
                return valid[choice]
            else:
                print("Invalid input. Try again.")
        except ValueError:
            print("Please enter a number.")


def p3_simulation_move(state):
    """
    Simulation mode: Player 3 uses Minimax (same as P1).
    """
    score, move = minimax(state, depth=3, is_maximizing=True,
                          current_player='p3', ai_player='p3')
    return move

print("Player 3 logic defined.")

Player 3 logic defined.


In [45]:
# Game tree printer

In [46]:
def print_game_tree(log, title="Game Tree"):
    """
    Prints the logged game tree from minimax or expectimax.
    """
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    # Print only first 60 lines to keep output manageable
    for line in log[:60]:
        print(line)
    if len(log) > 60:
        print(f"... ({len(log)-60} more lines) ...")
    print(f"{'='*60}\n")

print("Game tree printer defined.")

Game tree printer defined.


In [47]:
# main game loop

In [48]:
def check_winner(state):
    """
    Returns the winner's name if any player has 0 cards, else None.
    """
    if len(state['p1_hand']) == 0:
        return 'Player 1 (Minimax - Defensive)'
    if len(state['p2_hand']) == 0:
        return 'Player 2 (Expectimax - Offensive)'
    if len(state['p3_hand']) == 0:
        return 'Player 3'
    return None


def play_game(mode='simulation'):
    """
    Main game loop.
    mode: 'manual' (user plays as P3) or 'simulation' (all AI)
    """
    print("\n" + "="*60)
    print("         UNO GAME AI – Starting Game")
    print("="*60)

    state = initialize_game()
    turn = 0
    max_turns = 100  # Prevent infinite loop
    players = ['p1', 'p2', 'p3']
    player_names = {
        'p1': 'Player 1 (Minimax Defensive)',
        'p2': 'Player 2 (Expectimax Offensive)',
        'p3': 'Player 3'
    }

    # Logs for tree printing
    p1_log = []
    p2_log = []

    # Track if first tree was printed (to avoid printing every turn)
    p1_tree_printed = False
    p2_tree_printed = False

    while turn < max_turns:
        current = players[turn % 3]
        name = player_names[current]

        # Check skip
        if state.get('skip_next') and turn > 0:
            print(f"\n>> {name} is SKIPPED!")
            state['skip_next'] = False
            turn += 1
            continue

        print(f"\n{'─'*50}")
        print(f"Turn {turn+1} | {name}")
        print(f"Top Card: {state['top_card']}")
        print(f"P1 cards: {len(state['p1_hand'])} | P2 cards: {len(state['p2_hand'])} | P3 cards: {len(state['p3_hand'])}")
        print(f"Hand: {state[f'{current}_hand']}")

        # Decide move
        if current == 'p1':
            log = [] if not p1_tree_printed else None
            move = p1_minimax_move(state, log=log)
            if not p1_tree_printed and log:
                print_game_tree(log, "Minimax Game Tree (P1 – Turn 1 Sample)")
                p1_tree_printed = True

        elif current == 'p2':
            log = [] if not p2_tree_printed else None
            move = p2_expectimax_move(state, log=log)
            if not p2_tree_printed and log:
                print_game_tree(log, "Expectimax Game Tree (P2 – Turn 1 Sample)")
                p2_tree_printed = True

        else:  # p3
            if mode == 'manual':
                move = p3_manual_move(state)
            else:
                move = p3_simulation_move(state)

        # Display move
        if move:
            print(f">> {name} plays: {move}")
        else:
            print(f">> {name} DRAWS a card")

        # Apply move
        state = apply_move(state, current, move)

        # Check winner
        winner = check_winner(state)
        if winner:
            print(f"\n{'='*60}")
            print(f"  🏆 WINNER: {winner}!")
            print(f"{'='*60}")
            return winner

        turn += 1

    print("\nGame ended: Maximum turns reached (Draw).")
    return None

print("Game loop defined.")

Game loop defined.


In [49]:
# Run full simulation (all 3 players are AI)
winner = play_game(mode='simulation')


         UNO GAME AI – Starting Game

──────────────────────────────────────────────────
Turn 1 | Player 1 (Minimax Defensive)
Top Card: Yellow 9
P1 cards: 5 | P2 cards: 5 | P3 cards: 5
Hand: [Blue 4, Yellow 8, Green 2, Red Skip, Red 9]

  Minimax Game Tree (P1 – Turn 1 Sample)
[MAX] p1 | top=Yellow 9 | hand=[Blue 4, Yellow 8, Green 2, Red Skip, Red 9]
  -> Try: Yellow 8
    [MIN] p2 | top=Yellow 8 | hand=[Red 1, Yellow 6, Yellow 0, Red 7, Green 7]
      -> Try: Yellow 6
        [MIN] p3 | top=Yellow 6 | hand=[Red 8, Blue 1, Green 6, Green 3, Yellow 7]
          -> Try: Green 6
            [LEAF] depth=0 score=38.0
          -> Try: Yellow 7
            [LEAF] depth=0 score=38.0
          -> Try: DRAW
            [LEAF] depth=0 score=40.0
      -> Try: Yellow 0
        [MIN] p3 | top=Yellow 0 | hand=[Red 8, Blue 1, Green 6, Green 3, Yellow 7]
          -> Try: Yellow 7
            [LEAF] depth=0 score=38.0
          -> Try: DRAW
            [LEAF] depth=0 score=40.0
      -> Try: DRAW

---
## Cell 14: Run Manual Mode (Optional – Uncomment to Play)

In [50]:
# Uncomment below to play manually as Player 3
#winner = play_game(mode='manual')

In [51]:
#GUI

In [52]:
class UNO_GUI:
    """
    Tkinter-based GUI for UNO Game AI.
    Shows all hands, top card, and game log.
    Supports Manual (P3 clicks to play) and Simulation mode.
    """
    def __init__(self, root, mode='simulation'):
        self.root = root
        self.root.title("UNO Game AI – Assignment 2")
        self.root.geometry("900x680")
        self.root.configure(bg='#1a6b3c')  # UNO green background
        self.mode = mode

        # Color map for cards
        self.color_map = {
            'Red': '#e74c3c',
            'Blue': '#2980b9',
            'Green': '#27ae60',
            'Yellow': '#f1c40f'
        }

        self.state = initialize_game()
        self.turn = 0
        self.players = ['p1', 'p2', 'p3']
        self.player_names = {
            'p1': 'P1 Minimax (Defensive)',
            'p2': 'P2 Expectimax (Offensive)',
            'p3': 'P3 (You)' if mode == 'manual' else 'P3 Minimax'
        }
        self.game_over = False
        self.build_ui()
        self.update_display()

    def build_ui(self):
        """Build all UI widgets."""
        # Title
        tk.Label(self.root, text="UNO GAME AI", font=('Arial', 20, 'bold'),
                 bg='#1a6b3c', fg='white').pack(pady=8)

        # Top card area
        top_frame = tk.Frame(self.root, bg='#1a6b3c')
        top_frame.pack()
        tk.Label(top_frame, text="Top Card:", font=('Arial', 13, 'bold'),
                 bg='#1a6b3c', fg='white').pack(side='left', padx=5)
        self.top_card_label = tk.Label(top_frame, text="", font=('Arial', 14, 'bold'),
                                        width=12, height=2, relief='raised', borderwidth=3)
        self.top_card_label.pack(side='left', padx=10)

        # Scores frame
        score_frame = tk.Frame(self.root, bg='#1a6b3c')
        score_frame.pack(pady=4)
        self.score_labels = {}
        for p in self.players:
            lbl = tk.Label(score_frame, text="", font=('Arial', 11),
                           bg='#1a6b3c', fg='lightyellow', width=28)
            lbl.pack(side='left', padx=8)
            self.score_labels[p] = lbl

        # Hands frame
        hands_frame = tk.Frame(self.root, bg='#1a6b3c')
        hands_frame.pack(pady=6)

        self.hand_frames = {}
        for p in self.players:
            f = tk.LabelFrame(hands_frame, text=self.player_names[p],
                              font=('Arial', 10, 'bold'), bg='#145a32',
                              fg='white', padx=5, pady=5)
            f.pack(side='left', padx=10, pady=4, fill='both')
            self.hand_frames[p] = f

        # P3 play buttons (only shown in manual mode)
        self.p3_btn_frame = tk.Frame(self.root, bg='#1a6b3c')
        self.p3_btn_frame.pack(pady=4)
        self.p3_card_buttons = []

        # Control buttons
        ctrl_frame = tk.Frame(self.root, bg='#1a6b3c')
        ctrl_frame.pack(pady=6)

        self.next_btn = tk.Button(ctrl_frame, text="Next Turn ▶",
                                   font=('Arial', 12, 'bold'), bg='#f39c12',
                                   fg='white', width=14, command=self.next_turn)
        self.next_btn.pack(side='left', padx=8)

        tk.Button(ctrl_frame, text="New Game 🔄", font=('Arial', 12, 'bold'),
                  bg='#8e44ad', fg='white', width=14,
                  command=self.new_game).pack(side='left', padx=8)

        # Turn info
        self.turn_label = tk.Label(self.root, text="", font=('Arial', 12, 'bold'),
                                    bg='#1a6b3c', fg='white')
        self.turn_label.pack(pady=2)

        # Log box
        log_frame = tk.Frame(self.root, bg='#1a6b3c')
        log_frame.pack(fill='both', expand=True, padx=10, pady=4)
        tk.Label(log_frame, text="Game Log:", font=('Arial', 10, 'bold'),
                 bg='#1a6b3c', fg='white').pack(anchor='w')
        self.log_box = tk.Text(log_frame, height=8, font=('Courier', 9),
                               bg='#0d3b1e', fg='#00ff88', state='disabled')
        self.log_box.pack(fill='both', expand=True)

    def log(self, msg):
        """Append message to game log box."""
        self.log_box.configure(state='normal')
        self.log_box.insert('end', msg + '\n')
        self.log_box.see('end')
        self.log_box.configure(state='disabled')

    def make_card_btn(self, parent, card, command=None):
        """Create a colored card button widget."""
        bg = self.color_map.get(card.color, '#555')
        btn = tk.Button(parent, text=str(card.value), font=('Arial', 10, 'bold'),
                        bg=bg, fg='white', width=5, height=2,
                        relief='raised', borderwidth=2, command=command)
        return btn

    def update_display(self):
        """Refresh all UI elements to reflect current state."""
        # Update top card
        tc = self.state['top_card']
        tc_color = self.color_map.get(tc.color, '#888')
        self.top_card_label.config(text=f"{tc.color}\n{tc.value}",
                                    bg=tc_color, fg='white')

        # Update each player's hand
        for p in self.players:
            frame = self.hand_frames[p]
            for widget in frame.winfo_children():
                widget.destroy()

            hand = self.state[f'{p}_hand']
            for card in hand:
                bg = self.color_map.get(card.color, '#555')
                lbl = tk.Label(frame, text=f"{card.color[:1]}\n{card.value}",
                               font=('Arial', 9, 'bold'), bg=bg, fg='white',
                               width=4, height=2, relief='raised', borderwidth=2)
                lbl.pack(side='left', padx=2, pady=2)

            # Score labels
            score = evaluate(self.state, p, 'defensive' if p == 'p1' else 'offensive')
            self.score_labels[p].config(
                text=f"{self.player_names[p]}: {len(hand)} cards | Score: {score}")

        # Turn info
        current = self.players[self.turn % 3]
        self.turn_label.config(text=f"Turn {self.turn+1} → {self.player_names[current]}")

    def next_turn(self):
        """Execute the next player's turn."""
        if self.game_over:
            return

        current = self.players[self.turn % 3]
        name = self.player_names[current]

        # Handle skip
        if self.state.get('skip_next') and self.turn > 0:
            self.log(f"⏭ {name} is SKIPPED!")
            self.state['skip_next'] = False
            self.turn += 1
            self.update_display()
            return

        # Get move based on player
        if current == 'p1':
            move = p1_minimax_move(self.state)
        elif current == 'p2':
            move = p2_expectimax_move(self.state)
        else:
            if self.mode == 'manual':
                self.prompt_p3_move()
                return  # wait for user click
            else:
                move = p3_simulation_move(self.state)

        self.execute_move(current, name, move)

    def prompt_p3_move(self):
        """Show clickable buttons for P3 manual move."""
        for btn in self.p3_card_buttons:
            btn.destroy()
        self.p3_card_buttons = []

        hand = self.state['p3_hand']
        top = self.state['top_card']
        valid = get_valid_moves(hand, top)

        tk.Label(self.p3_btn_frame, text="Your move:",
                 font=('Arial', 10, 'bold'), bg='#1a6b3c', fg='white').pack(side='left')

        for card in valid:
            btn = self.make_card_btn(self.p3_btn_frame, card,
                                      command=lambda c=card: self.p3_play(c))
            btn.pack(side='left', padx=3)
            self.p3_card_buttons.append(btn)

        draw_btn = tk.Button(self.p3_btn_frame, text="DRAW",
                              font=('Arial', 10, 'bold'), bg='#7f8c8d',
                              fg='white', command=lambda: self.p3_play(None))
        draw_btn.pack(side='left', padx=3)
        self.p3_card_buttons.append(draw_btn)

    def p3_play(self, card):
        """Execute P3 manual move."""
        for btn in self.p3_card_buttons:
            btn.destroy()
        self.p3_card_buttons = []
        self.execute_move('p3', self.player_names['p3'], card)

    def execute_move(self, player, name, move):
        """Apply move, update display, check winner."""
        if move:
            self.log(f"Turn {self.turn+1} | {name} plays: {move}")
        else:
            self.log(f"Turn {self.turn+1} | {name} DRAWS a card")

        self.state = apply_move(self.state, player, move)
        self.turn += 1
        self.update_display()

        winner = check_winner(self.state)
        if winner:
            self.game_over = True
            self.log(f"\n🏆 WINNER: {winner}!")
            messagebox.showinfo("Game Over", f"Winner: {winner}!")
            self.next_btn.config(state='disabled')

    def new_game(self):
        """Reset and start a new game."""
        self.state = initialize_game()
        self.turn = 0
        self.game_over = False
        self.next_btn.config(state='normal')
        self.log_box.configure(state='normal')
        self.log_box.delete('1.0', 'end')
        self.log_box.configure(state='disabled')
        self.log("New game started!")
        self.update_display()


def launch_gui():
    """
    Launch the UNO GUI.
    Choose mode: 'simulation' or 'manual'
    """
    root = tk.Tk()
    # Ask user for mode
    mode = simpledialog.askstring("Game Mode",
                                   "Enter mode:\n'simulation' – All AI\n'manual' – You play as P3",
                                   initialvalue='simulation')
    if mode not in ['simulation', 'manual']:
        mode = 'simulation'
    app = UNO_GUI(root, mode=mode)
    root.mainloop()

# Launch GUI
launch_gui()


Top card: Red 4
P2 hand: [Blue 1, Green 0, Yellow 1, Green 5, Yellow 2]
P2 decision (All possible decisions at depth 1):
  DRAW | Expected score: 33.5

Top card: Red 6
P2 hand: [Blue 1, Green 0, Yellow 1, Green 5, Yellow 2, Red 9]
P2 decision (All possible decisions at depth 1):
  Play: Red 9 | Expected score: 38.5
  DRAW | Expected score: 28.5

Top card: Red 8
P2 hand: [Blue 1, Green 0, Yellow 1, Green 5, Yellow 2]
P2 decision (All possible decisions at depth 1):
  DRAW | Expected score: 30.5

Top card: Yellow 8
P2 hand: [Blue 1, Green 0, Yellow 1, Green 5, Yellow 2, Yellow 5]
P2 decision (All possible decisions at depth 1):
  Play: Yellow 1 | Expected score: 32.5
  Play: Yellow 2 | Expected score: 32.5
  Play: Yellow 5 | Expected score: 32.5
  DRAW | Expected score: 24.5

Top card: Yellow 1
P2 hand: [Blue 1, Green 0, Green 5, Yellow 2, Yellow 5]
P2 decision (All possible decisions at depth 1):
  Play: Blue 1 | Expected score: 40.5
  Play: Yellow 2 | Expected score: 40.5
  Play: Yell